# Wstęp do Uczenia Maszynowego - Projekt I
## Etap: Drugi Kamień Milowy 
### Autorzy: Krzysztof Osiński, Jakub Miszczak

## Import packages

In [ ]:
import pandas as pd
import numpy as np
import sklearn 
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib
import warnings
warnings.filterwarnings('ignore')
np.random.seed(23)
import zipfile

# Fraud Detection Transactions Dataset - Feature Extraction

In [ ]:
zip_path = "fraud-detection-transactions-dataset.zip"

with zipfile.ZipFile(zip_path, 'r') as z:
    with z.open("synthetic_fraud_dataset.csv") as file:
        df = pd.read_csv(file)

In [ ]:
df.columns = df.columns.str.replace(" ","_").str.lower()
df1 = df.drop(['transaction_id','user_id'],axis='columns')
df1['timestamp'] = pd.to_datetime(df1['timestamp'])

## Variance Inflation Factor (VIF)
Wartość VIF w okolicach 1 mówi o bardzo małej współliniowości cech.

In [ ]:
X = df1.drop('fraud_label', axis='columns')
Y = df1['fraud_label']

from sklearn.preprocessing import MinMaxScaler

scaled_columns = X.select_dtypes(['int64', 'float64']).columns

scaler = MinMaxScaler()

X[scaled_columns] = scaler.fit_transform(X[scaled_columns])
X.describe()

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

def calculate_vif(df):
    vif_df = pd.DataFrame()
    vif_df['Column'] = df.columns
    vif_df['VIF'] = [variance_inflation_factor(df.values,i) for i in range(df.shape[1])]
    return vif_df

In [ ]:
calculate_vif(X[scaled_columns])

## Weight of Evidence (WOE)

In [ ]:
# Funkcja do policzenia WOE i IV dla naszych cech
def calculate_woe_iv(df, feature, target):
    
    grouped = df.groupby(feature)[target].agg(['count','sum'])
    grouped = grouped.rename(columns={'count': 'total', 'sum': 'good'})
    grouped['bad']=grouped['total']-grouped['good']
    
    total_good = grouped['good'].sum()
    total_bad = grouped['bad'].sum()
    
    grouped['good_score'] = grouped['good'] / total_good
    grouped['bad_score'] = grouped['bad'] / total_bad
    grouped['woe'] = np.log(grouped['good_score']/ grouped['bad_score'])
    grouped['iv'] = (grouped['good_score'] -grouped['bad_score'])*grouped['woe']
    
    grouped['woe'] = grouped['woe'].replace([np.inf, -np.inf], 0)
    grouped['iv'] = grouped['iv'].replace([np.inf, -np.inf], 0)
    
    total_iv = grouped['iv'].sum()
    
    return grouped, total_iv

In [ ]:
iv_values = {}

# Liczymy IV 
for feature in X.columns:
    # Jeśli kolumna kategoryczna to względem kategorii
    if X[feature].dtype == 'object':
        _, iv = calculate_woe_iv(pd.concat([X, Y],axis=1), feature, 'fraud_label' )
    # Jeśli kolumna numeryczna to dzielimy na biny i liczymy względem binów
    else:
        X_binned = pd.cut(X[feature], bins=10, labels=False)
        _, iv = calculate_woe_iv(pd.concat([X_binned, Y],axis=1), feature, 'fraud_label' )
    iv_values[feature] = iv
    
# Eleganckie zapisanie wyników 
pd.set_option('display.float_format', lambda x: '{:.4f}'.format(x))

iv_df = pd.DataFrame(list(iv_values.items()), columns=['Feature', 'IV'])
iv_df = iv_df.sort_values(by='IV', ascending=False)

## Information Value (IV)
IV < 0.02  --> brak wartości predykcyjnej

0.3 <= IV < 0.5  --> silna wartość predykcyjna

IV >= 0.5  --> bardzo silna wartość predykcyjna (trzeba uważać na overfitting)

In [ ]:
iv_df

### Spostrzeżenia
Na tym etapie wartość predykcyjną wykazują jedynie cechy 'failed_transaction_count_7d' i 'risk_score'.

Teraz poprzez interakcje między zmiennymi i transformacje zmiennych spróbujmy znaleźć więcej cech znaczących.

In [ ]:
# Tworzenie nowych cech
X['failed_tx_risk_interaction'] = X['failed_transaction_count_7d'] * X['risk_score']
X['failed_tx_ratio'] = X['failed_transaction_count_7d'] / (X['daily_transaction_count'] + 1)
X['amount_risk_interaction'] = X['transaction_amount'] * X['risk_score']
X['balance_to_avg_tx_ratio'] = X['account_balance'] / (X['avg_transaction_amount_7d'] + 1)

# Transformacje matematyczne
X['log_transaction_amount'] = np.log1p(X['transaction_amount'])
X['log_account_balance'] = np.log1p(X['account_balance'])
X['sqrt_risk_score'] = np.sqrt(X['risk_score'])
X['sqrt_transaction_distance'] = np.sqrt(X['transaction_distance'])

# Ekstrakcja cech czasowych
X['transaction_hour'] = X['timestamp'].dt.hour
X['transaction_dayofweek'] = X['timestamp'].dt.dayofweek

scaled_columns = X.select_dtypes(['int64', 'float64']).columns

In [ ]:
iv_values = {}
for feature in scaled_columns:
    X_binned = pd.cut(X[feature], bins=10, labels=False)
    _, iv = calculate_woe_iv(pd.concat([X_binned, Y], axis=1), feature, 'fraud_label')
    iv_values[feature] = iv

iv_df = pd.DataFrame(list(iv_values.items()), columns=['Feature', 'IV'])
iv_df = iv_df.sort_values(by='IV', ascending=False)

iv_df

Widzimy tutaj, że niektóre z nowo stworzonych cech wykazują bardzo satysfakconujące wskaźniki korelacji IV.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns


df1['sqrt_risk_score'] = np.sqrt(df1['risk_score'])
df1['failed_tx_risk_interaction'] = df1['failed_transaction_count_7d'] * df1['risk_score']
df1['failed_tx_ratio'] = df1['failed_transaction_count_7d'] / (df1['failed_transaction_count_7d'] + df1['daily_transaction_count'])

# Lista cech do analizy
features = ['sqrt_risk_score', 'failed_tx_risk_interaction', 'failed_tx_ratio',
            'failed_transaction_count_7d', 'risk_score']

### Stwórzmy kilka prostych wykresów żeby zobaczyć jak wyglądają te korelacje.

### Boxploty:

In [ ]:
plt.figure(figsize=(15, 12))

# Boxploty
for i, feature in enumerate(features, 1):
    plt.subplot(3, 2, i)
    sns.boxplot(data=df1, x='fraud_label', y=feature)
    plt.title(f'Boxplot: {feature} vs Fraud Label')

plt.tight_layout()
plt.show()

### Histogramy:

In [ ]:
plt.figure(figsize=(15, 12))

# Histogramy
for i, feature in enumerate(features, 1):
    plt.subplot(3, 2, i)
    sns.histplot(data=df1, x=feature, hue='fraud_label', kde=True, bins=30, alpha=0.5)
    plt.title(f'Histogram: {feature} vs Fraud Label')

plt.tight_layout()
plt.show()

### Wykresy KDE:

In [ ]:
plt.figure(figsize=(15, 12))

# KDE Plots
for i, feature in enumerate(features, 1):
    plt.subplot(3, 2, i)
    sns.kdeplot(data=df1[df1['fraud_label'] == 0], x=feature, label='Not Fraud', fill=True, alpha=0.5)
    sns.kdeplot(data=df1[df1['fraud_label'] == 1], x=feature, label='Fraud', fill=True, alpha=0.5)
    plt.title(f'KDE Plot: {feature} vs Fraud Label')
    plt.legend()

plt.tight_layout()
plt.show()